[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C55_TSR_Autonomous_Driving_Course/05_safety_eval/05_safety_oriented_eval.ipynb)

# 05 · 安全导向的评测体系（分桶 mAP / 代价敏感风险分 / FP-per-km / 时序稳定性 / 回归门禁）

目标：把「mAP 0.85」这种**不足以做决策**的数字，换成一整套**能拦住安全回退**的指标体系，
并把它做成一条可复现、可测试的评测流水线。

本 notebook 你会亲手实现：

1. 从零实现 **AP**（all-point 插值），并用手算过的例子做单元测试
2. **分桶评测**：构造一对「整体 mAP 几乎相同、小目标桶差 0.3」的模型，
   并展示 **micro / macro 平均会给出相反的排名**
3. **代价敏感风险分**：构造 mAP 更高但风险分高 1.8 倍的模型 —— 用 mAP 选型会选错
4. **FP per km / per hour**、**per-sign recall vs per-frame recall**，以及工作点选择
5. **时序稳定性指标**：首次上报距离 / 稳定检出距离 / 闪烁次数 / 误报持续时长
6. **回归门禁判定器**：两比例检验 + **Benjamini–Hochberg 多重比较校正** + 三档门禁
7. **离线-路测背离的量化**：分母不匹配到底让你低估了多少倍

> 心智模型：**mAP 是迭代信号，不是发布判据。
> 评测流水线的产物不是数字，是「能不能发」这个决策。**

## 1 · 从零实现 AP，并给它写单元测试

**评测代码本身必须有单元测试** —— 指标算错不会报错，只会让所有决策基于错误的数字。
三组标准输入：完美预测（应得 1.0）、全错预测（应得 0.0）、手算过的小例子。

In [ ]:
import numpy as np, math, itertools
from collections import Counter, defaultdict
rng = np.random.default_rng(5)

def iou(a, b):
    x1 = max(a[0], b[0]); y1 = max(a[1], b[1])
    x2 = min(a[2], b[2]); y2 = min(a[3], b[3])
    iw = max(0.0, x2 - x1); ih = max(0.0, y2 - y1)
    inter = iw * ih
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / ua if ua > 0 else 0.0

def average_precision(scores, tp, n_gt):
    '''all-point 插值 AP（COCO 风格）。scores/tp 同序，tp 为 0/1。'''
    if n_gt == 0:
        return float('nan')
    if len(scores) == 0:
        return 0.0
    order = np.argsort(-np.asarray(scores, dtype=float), kind='stable')
    t = np.asarray(tp, dtype=float)[order]
    ctp = np.cumsum(t); cfp = np.cumsum(1.0 - t)
    rec = ctp / n_gt
    prec = ctp / np.maximum(ctp + cfp, 1e-12)
    mrec = np.concatenate([[0.0], rec, [rec[-1]]])
    mpre = np.concatenate([[0.0], prec, [0.0]])
    for i in range(len(mpre) - 2, -1, -1):          # 单调递减包络
        mpre[i] = max(mpre[i], mpre[i+1])
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[idx+1] - mrec[idx]) * mpre[idx+1]))

# ── 单元测试三件套 ──
assert average_precision([0.9, 0.8, 0.7], [1, 1, 1], 3) == 1.0, '完美预测应得 1.0'
assert average_precision([0.9, 0.8, 0.7], [0, 0, 0], 3) == 0.0, '全错应得 0.0'
# 手算例子：n_gt=3，按分数降序 tp = [1,0,1,1]
#   rec = [1/3,1/3,2/3,1]  prec = [1,.5,.667,.75]
#   包络后 mpre = [1,1,.75,.75,.75,0]
#   AP = 1/3*1 + 1/3*.75 + 1/3*.75 = 0.833333
ap_hand = average_precision([0.9, 0.8, 0.7, 0.6], [1, 0, 1, 1], 3)
assert abs(ap_hand - 5/6) < 1e-9, f'手算应为 0.833333，得到 {ap_hand}'
assert abs(average_precision([0.9], [1], 3) - 1/3) < 1e-9, '只召回 1/3 且 precision=1'
print(f'手算例子 AP = {ap_hand:.6f}   ✅ AP 实现通过三组单元测试')
print('⚠️  评测代码的 bug 不会报错，只会让所有决策基于错误的数字 —— 必须有单测。')

In [ ]:
# ── 逐帧框匹配（按分数降序贪心，一个 GT 只能被认领一次）──
def match_frame(gt_boxes, dets, iou_thr=0.5):
    '''dets: [(box, score), ...]。返回与 dets 同序的 tp 标记。'''
    order = sorted(range(len(dets)), key=lambda i: -dets[i][1])
    used = [False] * len(gt_boxes)
    tp = [0] * len(dets)
    for i in order:
        box = dets[i][0]
        best, best_j = iou_thr, -1
        for j, g in enumerate(gt_boxes):
            if used[j]:
                continue
            v = iou(box, g)
            if v >= best:
                best, best_j = v, j
        if best_j >= 0:
            used[best_j] = True; tp[i] = 1
    return tp

G = [np.array([100., 100., 140., 140.]),
     np.array([300., 300., 320., 320.]),
     np.array([700., 400., 716., 416.])]
Dets = [(np.array([102., 101., 142., 141.]), 0.95),   # 命中 G0
        (np.array([104., 103., 144., 143.]), 0.80),   # **重复检测** G0 -> FP
        (np.array([301., 300., 321., 320.]), 0.70),   # 命中 G1
        (np.array([900., 900., 930., 930.]), 0.60),   # 背景 FP
        (np.array([702., 402., 718., 418.]), 0.40)]   # 命中 G2，但 IoU 只有 0.62
tp = match_frame(G, Dets, iou_thr=0.5)
print('tp 标记:', tp, '  (第 2 个是重复检测 -> FP)')
assert tp == [1, 0, 1, 0, 1], tp
ap = average_precision([d[1] for d in Dets], tp, len(G))
print(f'该帧 AP@0.5 = {ap:.4f}')

# ── 顺手量化一个对 TSR 至关重要的事实：**同样的定位误差，小框吃亏得多** ──
print(f"\n{'框边长':>8s} {'偏移 2px':>10s} {'偏移 5px':>10s} {'偏移 8px':>10s}")
for s in [16, 40, 96]:
    a = np.array([0., 0., float(s), float(s)])
    row = [iou(a, np.array([d, d, s+d, s+d])) for d in (2., 5., 8.)]
    print(f'{s:>7d}px ' + ' '.join(f'{v:>10.3f}' for v in row))
assert iou(np.array([0.,0.,16.,16.]), np.array([5.,4.,21.,20.])) < 0.5
assert iou(np.array([0.,0.,40.,40.]), np.array([5.,4.,45.,44.])) > 0.6
print('\n⚠️  **16px 的框偏 (5,4)px 掉到 IoU 0.35（判为「漏检 + 误检」两笔账），')
print('    而 40px 的框偏同样距离仍有 0.65（判为命中）。**')
print('   同一个 IoU 阈值对不同尺度是极不公平的（C57 会严格推导这一点）——')
print('✅ 直接后果：**TSR 的评测必须按像素尺寸分桶**，否则「远处定位差」这件事')
print('   会被记成「远处漏检」，你连改哪儿都定位不到。')

# COCO mAP@[.5:.95] 对定位精度的惩罚有多重？
def ap_over_iou_range(gt_boxes, dets, thrs=None):
    thrs = thrs if thrs is not None else [0.50 + 0.05*i for i in range(10)]
    return float(np.mean([average_precision([d[1] for d in dets],
                                            match_frame(gt_boxes, dets, t),
                                            len(gt_boxes)) for t in thrs]))
print(f'该帧 mAP@[.5:.95] = {ap_over_iou_range(G, Dets):.4f}  '
      f'（比 AP@0.5 低很多 —— 一半权重压在 IoU>0.75 上）')
assert ap_over_iou_range(G, Dets) < ap
print('\n⚠️  对 TSR 而言，把框画准 0.05 个 IoU **没有任何下游价值** ——')
print('   下游要的是「限速值是多少」和「它管哪段路」。')
print('✅ 所以 TSR 的主指标不该是 mAP@[.5:.95]，而应是代价加权的语义指标（第 3 节）。')

## 2 · 分桶评测：平均值会说谎

构造一对模型：**整体 AP 几乎相同，但小目标桶差 0.3**。
再看 micro（按实例）与 macro（按桶）平均如何给出**相反的排名**。

In [ ]:
BUCKETS = [('<16px', 0, 16), ('16-32px', 16, 32), ('>=32px', 32, 1e9)]
N_PER_BUCKET = {'<16px': 100, '16-32px': 300, '>=32px': 600}
# 两个模型在各桶的召回上限（刻意设计：A 小目标强、B 大目标强）
RECALL = {'A': {'<16px': 0.80, '16-32px': 0.90, '>=32px': 0.92},
          'B': {'<16px': 0.50, '16-32px': 0.95, '>=32px': 0.96}}

def bucket_of(size_px):
    for name, lo, hi in BUCKETS:
        if lo <= size_px < hi:
            return name
    return BUCKETS[-1][0]

def make_gt(seed=0):
    g = np.random.default_rng(seed)
    gts = []
    for name, lo, hi in BUCKETS:
        hi_ = 96.0 if hi > 1e8 else hi
        for _ in range(N_PER_BUCKET[name]):
            gts.append({'bucket': name, 'size': float(g.uniform(lo + 1, hi_))})
    return gts

def make_preds(gts, model, seed=1, n_fp=200):
    '''每个 GT 按桶召回率被检出，分数从高分布采；再加一批与模型无关的 FP。'''
    g = np.random.default_rng(seed)
    recs = []
    for i, gt in enumerate(gts):
        if g.random() < RECALL[model][gt['bucket']]:
            recs.append({'bucket': gt['bucket'], 'tp': 1,
                         'score': float(np.clip(g.normal(0.75, 0.15), 0.01, 0.999))})
    gfp = np.random.default_rng(999)                    # **两个模型共用同一批 FP**
    for _ in range(n_fp):
        size = float(gfp.uniform(6, 96))
        recs.append({'bucket': bucket_of(size), 'tp': 0,
                     'score': float(np.clip(gfp.normal(0.35, 0.15), 0.01, 0.999))})
    return recs

gts = make_gt()
preds = {m: make_preds(gts, m, seed=2) for m in ['A', 'B']}

def ap_of(recs, n_gt):
    return average_precision([r['score'] for r in recs], [r['tp'] for r in recs], n_gt)

def report(m):
    recs = preds[m]
    per_bucket = {}
    for name, _, _ in BUCKETS:
        sub = [r for r in recs if r['bucket'] == name]
        per_bucket[name] = ap_of(sub, N_PER_BUCKET[name])
    overall = ap_of(recs, len(gts))
    macro = float(np.mean(list(per_bucket.values())))
    return per_bucket, overall, macro

print(f"{'模型':>5s} " + ' '.join(f'{n:>10s}' for n, _, _ in BUCKETS) +
      f" {'整体 AP':>9s} {'macro':>8s}")
res = {}
for m in ['A', 'B']:
    pb, ov, ma = report(m); res[m] = (pb, ov, ma)
    print(f'{m:>5s} ' + ' '.join(f'{pb[n]:>10.3f}' for n, _, _ in BUCKETS) +
          f' {ov:>9.3f} {ma:>8.3f}')

pbA, ovA, maA = res['A']; pbB, ovB, maB = res['B']
assert abs(ovA - ovB) < 0.06, f'整体 AP 应几乎相同：{ovA:.3f} vs {ovB:.3f}'
assert pbA['<16px'] - pbB['<16px'] > 0.15, '小目标桶应差一大截'
assert maA > maB, 'macro 平均应给出与整体 AP 相反的排名'
print(f'\n整体 AP：A={ovA:.3f} vs B={ovB:.3f}  →  差 {abs(ovA-ovB):.3f}（噪声量级）')
print(f'小目标桶：A={pbA["<16px"]:.3f} vs B={pbB["<16px"]:.3f}  →  差 {pbA["<16px"]-pbB["<16px"]:.3f}（灾难级）')
print(f'macro   ：A={maA:.3f} vs B={maB:.3f}  →  **排名与整体 AP 相反**')
print('\n⚠️  micro（按实例）被样本量最大的桶主导，而那通常是**最容易的一档**。')
print('✅ TSR 的正确默认是 macro，并对安全关键桶再加权。')

In [ ]:
# ── 分桶的统计代价：桶越细，置信区间越宽 ──
def bootstrap_ci(recs, n_gt, n_boot=400, alpha=0.05, seed=0):
    '''对「实例」做 bootstrap 重采样，给出 AP 的置信区间。'''
    g = np.random.default_rng(seed)
    idx = np.arange(len(recs))
    vals = []
    for _ in range(n_boot):
        take = g.choice(idx, size=len(idx), replace=True)
        sub = [recs[i] for i in take]
        vals.append(ap_of(sub, n_gt))
    lo, hi = np.percentile(vals, [100*alpha/2, 100*(1-alpha/2)])
    # 对实例重采样时 TP 数可能超过 n_gt（真值数是固定的），故把区间裁回 [0,1]
    return float(np.clip(lo, 0.0, 1.0)), float(np.clip(hi, 0.0, 1.0))

print(f"{'桶':>10s} {'正样本数':>9s} {'AP(A)':>8s} {'95% CI':>18s} {'CI 宽度':>9s}")
for name, _, _ in BUCKETS:
    sub = [r for r in preds['A'] if r['bucket'] == name]
    n_gt = N_PER_BUCKET[name]
    lo, hi = bootstrap_ci(sub, n_gt, seed=3)
    print(f'{name:>10s} {n_gt:>9d} {pbA[name]:>8.3f} '
          f'{f"[{lo:.3f}, {hi:.3f}]":>18s} {hi-lo:>9.3f}')

# 再切一个只有 30 个正样本的「关键类」桶（按比例下采样，TP 与 FP 都保留）
g_tiny = np.random.default_rng(21)
tiny = [r for r in preds['A'] if r['bucket'] == '<16px' and g_tiny.random() < 0.30]
lo_t, hi_t = bootstrap_ci(tiny, 30, seed=4)
print(f'\n只有 30 个正样本的桶：AP={ap_of(tiny,30):.3f}  95% CI=[{lo_t:.3f}, {hi_t:.3f}]'
      f'  宽度={hi_t-lo_t:.3f}')
w_small = bootstrap_ci([r for r in preds['A'] if r['bucket']=='<16px'], 100, seed=3)
w_large = bootstrap_ci([r for r in preds['A'] if r['bucket']=='>=32px'], 600, seed=3)
assert (w_small[1]-w_small[0]) > (w_large[1]-w_large[0]), '小样本桶的 CI 必然更宽'
assert (hi_t - lo_t) > 0.10, '30 个正样本的桶，CI 宽度超过 0.10'
print('\n⚠️  30 个正样本的桶，AP 的 95% CI 宽度 > 0.10 —— 「掉了 0.05」根本不是信号。')
print('✅ 两条硬规则：① 每桶 >= 50 个正样本  ② **所有分桶指标必须带 bootstrap CI**。')

## 3 · 代价敏感风险分：用 mAP 选型会选错

三层不对称：类别之间、错误方向之间、漏检与误检之间。
把它们写成代价矩阵，风险分就替代 mAP 成为**唯一主指标**。

In [ ]:
CLASSES = ['stop_yield', 'no_entry', 'speed_60', 'speed_80', 'speed_120', 'info']
GT_COUNT = {'stop_yield': 40, 'no_entry': 30, 'speed_60': 300,
            'speed_80': 300, 'speed_120': 200, 'info': 500}

COST_MISS = {'stop_yield': 100.0, 'no_entry': 80.0, 'speed_60': 8.0,
             'speed_80': 8.0, 'speed_120': 6.0, 'info': 0.5}
COST_FP   = {'stop_yield': 20.0, 'no_entry': 25.0, 'speed_60': 12.0,
             'speed_80': 12.0, 'speed_120': 10.0, 'info': 0.5}
# 错分代价**方向不对称**：低值错成高值 = 超速（危险）；高值错成低值 = 保守
COST_CONF = {('speed_60', 'speed_80'): 15.0, ('speed_80', 'speed_60'): 2.0,
             ('speed_80', 'speed_120'): 15.0, ('speed_120', 'speed_80'): 2.0,
             ('speed_60', 'speed_120'): 25.0, ('speed_120', 'speed_60'): 3.0}

def conf_cost(c_true, c_pred):
    if (c_true, c_pred) in COST_CONF:
        return COST_CONF[(c_true, c_pred)]
    return max(COST_MISS[c_true], COST_FP[c_pred]) * 0.6   # 跨组错分：取较重的一侧打折

print(f"{'错误':<34s} {'代价':>7s}")
for k, v in [('漏检 stop_yield', COST_MISS['stop_yield']),
             ('漏检 no_entry', COST_MISS['no_entry']),
             ('speed_60 -> speed_80（超速）', COST_CONF[('speed_60','speed_80')]),
             ('speed_80 -> speed_60（保守）', COST_CONF[('speed_80','speed_60')]),
             ('误检一块不存在的 speed_60', COST_FP['speed_60']),
             ('漏检 info', COST_MISS['info'])]:
    print(f'{k:<34s} {v:>7.1f}')
assert COST_CONF[('speed_60','speed_80')] > 5 * COST_CONF[('speed_80','speed_60')]
print('\n⚠️  **同一对类别，两个方向的代价差 7.5 倍** —— 这是最常被忽略的一层不对称。')
print('   代价数值不能拍脑袋：正规做法是从 HARA / FMEA 反推，有文档、有评审、有版本号。')

In [ ]:
# ── 构造两个模型：A 的 mAP 更高，但风险分是 B 的 1.8 倍 ──
RECALL_C = {
    'A': {'stop_yield': 0.70, 'no_entry': 0.75, 'speed_60': 0.98,
          'speed_80': 0.98, 'speed_120': 0.97, 'info': 0.99},
    'B': {'stop_yield': 0.95, 'no_entry': 0.93, 'speed_60': 0.91,
          'speed_80': 0.89, 'speed_120': 0.86, 'info': 0.82},
}
# 错分计数（GT 类 -> 预测类），只列限速类之间
CONFUSE = {'A': {('speed_60','speed_80'): 9, ('speed_80','speed_60'): 4,
                 ('speed_80','speed_120'): 5, ('speed_120','speed_80'): 3},
           'B': {('speed_60','speed_80'): 3, ('speed_80','speed_60'): 6,
                 ('speed_80','speed_120'): 2, ('speed_120','speed_80'): 5}}
FP_COUNT = {'A': {'stop_yield': 1, 'no_entry': 1, 'speed_60': 6,
                  'speed_80': 6, 'speed_120': 4, 'info': 30},
            'B': {'stop_yield': 2, 'no_entry': 1, 'speed_60': 5,
                  'speed_80': 5, 'speed_120': 4, 'info': 25}}
DIST_KM = 100.0

def risk_score(model, dist_km=DIST_KM):
    '''代价加权风险分（每 100 km）。'''
    r = 0.0
    for c in CLASSES:
        n_miss = GT_COUNT[c] * (1.0 - RECALL_C[model][c])
        r += n_miss * COST_MISS[c]
        r += FP_COUNT[model][c] * COST_FP[c]
    for (ct, cp), n in CONFUSE[model].items():
        r += n * conf_cost(ct, cp)
    return r / dist_km * 100.0

def macro_recall(model):
    return float(np.mean([RECALL_C[model][c] for c in CLASSES]))

print(f"{'模型':>5s} {'macro recall(≈mAP 代理)':>24s} {'风险分 / 100km':>16s}")
for m in ['A', 'B']:
    print(f'{m:>5s} {macro_recall(m):>24.4f} {risk_score(m):>16.1f}')

mrA, mrB = macro_recall('A'), macro_recall('B')
rA, rB = risk_score('A'), risk_score('B')
print(f'\nmAP 代理: A 比 B 高 {mrA-mrB:+.4f}（**在种子噪声范围内**，会被判为「无差别」甚至「A 更好」）')
print(f'风险分  : A 是 B 的 {rA/rB:.2f} 倍  ←  **A 是一次严重的安全回退**')
assert mrA > mrB, 'A 的平均召回更高'
assert rA > 1.5 * rB, f'A 的风险分应显著更高：{rA:.1f} vs {rB:.1f}'
print('\n🚨 如果发布门禁只看 mAP，A 会被放行 —— 而它在 stop_yield 上掉了 25 个点。')
print('✅ 风险分天然把关键类的权重提上来，量纲有物理意义（每 100 km 承担多少风险），')
print('   可以跨版本、跨车型比较，且对 micro/macro 之争免疫。')

# 逐项归因：风险分的钱花在哪里
print(f"\n{'类别':<12s} {'A 漏检代价':>11s} {'B 漏检代价':>11s} {'差额':>9s}")
for c in CLASSES:
    ca = GT_COUNT[c]*(1-RECALL_C['A'][c])*COST_MISS[c]
    cb = GT_COUNT[c]*(1-RECALL_C['B'][c])*COST_MISS[c]
    print(f'{c:<12s} {ca:>11.0f} {cb:>11.0f} {ca-cb:>+9.0f}')
print('✅ 风险分不只是排序，还能**逐项归因** —— 直接告诉你下一步该修哪个类。')

## 4 · 换分母：FP per km 与 per-sign recall

判据：**分母是系统自己的输出（precision）→ 可被优化行为污染；
分母是外生物理量（里程、物理标志数）→ 诚实。**

In [ ]:
# ── precision 可以靠提阈值刷高，FP/km 不能 ──
AVG_SPEED_KMH = 60.0
FPS_EVAL = 30.0
DRIVE_KM = 100.0
DRIVE_HOURS = DRIVE_KM / AVG_SPEED_KMH
N_FRAMES = DRIVE_HOURS * 3600 * FPS_EVAL

recs = preds['A']
n_gt_all = len(gts)
print(f'行驶 {DRIVE_KM:.0f} km（{DRIVE_HOURS:.2f} h，{N_FRAMES:.0f} 帧）\n')
print(f"{'阈值':>6s} {'检出数':>7s} {'TP':>6s} {'FP':>6s} {'precision':>10s} "
      f"{'recall':>8s} {'FP/km':>8s} {'FP/h':>8s}")
rows = []
for thr in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    kept = [r for r in recs if r['score'] >= thr]
    tp_ = sum(r['tp'] for r in kept); fp_ = len(kept) - tp_
    prec = tp_ / len(kept) if kept else 1.0
    rec_ = tp_ / n_gt_all
    fp_km = fp_ / DRIVE_KM; fp_h = fp_ / DRIVE_HOURS
    rows.append((thr, prec, rec_, fp_km))
    print(f'{thr:>6.1f} {len(kept):>7d} {tp_:>6d} {fp_:>6d} {prec:>10.4f} '
          f'{rec_:>8.4f} {fp_km:>8.3f} {fp_h:>8.2f}')

precs = [r[1] for r in rows]; recs_ = [r[2] for r in rows]; fpkms = [r[3] for r in rows]
assert precs == sorted(precs), 'precision 随阈值单调上升 —— **它可以被刷**'
assert recs_ == sorted(recs_, reverse=True), 'recall 随阈值单调下降'
assert fpkms == sorted(fpkms, reverse=True), 'FP/km 随阈值单调下降'
print('\n⚠️  precision 从 %.3f 涨到 %.3f 看起来是「进步」，实际只是砍掉了一半召回。'
      % (precs[0], precs[-1]))
print('✅ FP/km 与 recall **一起动**，取舍暴露出来，没法自欺欺人。')
print('   量级感：FP/km = 0.1 → 每 10 km 一次误报 → 30 km 通勤每天 3 次 → 用户一周内关掉功能。')
print('   量产目标通常是 **FP per 100 km < 1**，触发实际控制动作的关键类还要再严 1-2 个数量级。')

In [ ]:
# ── 工作点选择：在 FP 预算约束下最大化 recall ──
def select_operating_point(rows, fp_per_100km_budget):
    ok = [r for r in rows if r[3] * 100.0 <= fp_per_100km_budget]
    return max(ok, key=lambda r: r[2]) if ok else None

for budget in [500, 100, 30, 10]:
    op = select_operating_point(rows, budget)
    if op:
        print(f'FP/100km 预算 {budget:>4d} → 阈值 {op[0]:.1f}, '
              f'recall {op[2]:.4f}, FP/100km {op[3]*100:.1f}')
    else:
        print(f'FP/100km 预算 {budget:>4d} → **无可行工作点**（需要改模型，不是改阈值）')
op_loose = select_operating_point(rows, 500)
op_tight = select_operating_point(rows, 100)
assert op_loose[2] >= op_tight[2], '预算越紧，可达 recall 越低'
print('\n✅ 「阈值取多少」不该由算法工程师拍，而应由 **FP 预算**（产品/安全给的）反解。')

# ── per-frame recall vs per-sign recall ──
def per_sign_metrics(det_flags, ranges):
    '''det_flags: 每帧是否检出；ranges: 每帧对应的距离(m)。'''
    n = len(det_flags)
    per_frame = sum(det_flags) / n
    per_sign = 1.0 if any(det_flags) else 0.0
    first = max((r for f, r in zip(det_flags, ranges) if f), default=None)
    return per_frame, per_sign, first

n_fr = 90
ranges = list(np.linspace(100.0, 15.0, n_fr))
flags = [False]*40 + [True, True] + [False, False] + [True]*4 + [False]*2 + \
        [True, True] + [False] + [True]*4 + [False]*(n_fr - 55)
flags = (flags + [False]*n_fr)[:n_fr]
pf, ps, fr = per_sign_metrics(flags, ranges)
print(f'\n一块牌子可见 {n_fr} 帧，检出 {sum(flags)} 帧：')
print(f'  per-frame recall = {pf:.3f}   ← 看起来像个失败的系统')
print(f'  per-sign  recall = {ps:.3f}   ← 用户视角：这块牌子被识别了')
print(f'  首次检出距离     = {fr:.1f} m  ← 真正决定体验的量')
assert abs(pf - sum(flags)/n_fr) < 1e-9 and ps == 1.0 and pf < 0.2
print('\n⚠️  只报 per-frame recall，会让你去优化一个不重要的量（每帧都要检出）。')
print('⚠️  只报 per-sign recall，会掩盖「15 米才第一次检出」这种来不及减速的情况。')
print('✅ 正确的一组：per-sign recall（有没有）+ 首检距离 p50/p90（多远）+ 稳定检出距离（从哪起可信）。')

## 5 · 时序稳定性指标：mAP 完全看不见的那一半

五个指标：首次**上报**距离 / 稳定检出距离 / 闪烁次数 / 误报持续时长 p95 / 类别跳变率。
注意是「上报」不是「检出」—— 只有测上报，模块 04 的参数才有目标函数。

In [ ]:
def stability_metrics(report_flags, ranges, stable_ratio=0.95):
    '''report_flags: 每帧是否**上报**；ranges: 每帧距离（单调递减）。'''
    n = len(report_flags)
    first_range = max((r for f, r in zip(report_flags, ranges) if f), default=None)
    # 稳定检出距离：最远的起点 i，使得 [i, n) 内上报比例 >= stable_ratio
    stable_range, suffix = None, 0
    for i in range(n - 1, -1, -1):
        suffix += report_flags[i]
        if suffix / (n - i) >= stable_ratio:
            stable_range = ranges[i]
    flips = sum(1 for i in range(1, n) if report_flags[i] != report_flags[i-1])
    return {'first_range': first_range, 'stable_range': stable_range, 'flips': flips}

def fp_episodes(flags, fps=30.0):
    '''把连续的 FP 帧合并成「事件」，返回每个事件的时长（秒）。'''
    eps, run = [], 0
    for f in list(flags) + [False]:
        if f:
            run += 1
        elif run:
            eps.append(run / fps); run = 0
    return eps

# 三个系统：同样的单帧检测器，不同的时序策略（模块 04 的三种配置）
det = ([False]*30                                     # 太远，检不到
       + [True, False, True, True, False, True]       # 30-35 帧：远距离断续检出
       + [True, True, False, True, True, False, False, True, True, True]   # 36-45：仍有抖动
       + [True]*44)                                   # 46 帧起稳定检出
rng_ = list(np.linspace(100.0, 15.0, len(det)))
sys_raw   = det                                                # 无时序处理
sys_vote  = [sum(det[max(0,i-4):i+1]) >= 3 for i in range(len(det))]      # 3-of-5
sys_hyst  = []                                                 # 3-of-5 + 迟滞(撤销要连续 4 帧 miss)
on, miss = False, 0
for i in range(len(det)):
    w = sum(det[max(0, i-4):i+1])
    miss = 0 if det[i] else miss + 1
    on = (w >= 3) if not on else (miss < 4)
    sys_hyst.append(on)

print(f"{'系统':<20s} {'首次上报':>10s} {'稳定检出':>10s} {'状态翻转':>9s}")
for name, s in [('单帧（无时序）', sys_raw), ('3-of-5 投票', sys_vote),
                ('3-of-5 + 迟滞', sys_hyst)]:
    m = stability_metrics(s, rng_)
    print(f'{name:<20s} {m["first_range"]:>9.1f}m {m["stable_range"]:>9.1f}m {m["flips"]:>9d}')

m_raw = stability_metrics(sys_raw, rng_)
m_vote = stability_metrics(sys_vote, rng_)
m_hyst = stability_metrics(sys_hyst, rng_)
assert m_raw['first_range'] > m_vote['first_range'], '投票让首报距离后退（延迟的代价）'
assert m_hyst['flips'] < m_raw['flips'], '迟滞把翻转次数压下来'
assert m_hyst['flips'] <= m_vote['flips'], '迟滞不应比纯投票更抖'
assert m_hyst['stable_range'] >= m_vote['stable_range'], '迟滞让「可信起点」更远'
print('\n⚠️  **这三行的 mAP 完全相同** —— 逐帧的框和分数一模一样，只是时序策略不同。')
print('   mAP 对模块 04 的全部工作一分收益都不给。')

# 误报持续时长：迟滞的反面账单
fp_raw  = [False]*20 + [True, False, True] + [False]*20 + [True] + [False]*20
fp_hyst = [False]*20 + [True]*8 + [False]*15 + [True]*6 + [False]*15
e_raw, e_hyst = fp_episodes(fp_raw), fp_episodes(fp_hyst)
print(f'\n误报事件（帧级 FP 合并成事件）:')
print(f'  无迟滞: {len(e_raw)} 次, 时长 {[round(x,3) for x in e_raw]} s, p95={np.percentile(e_raw,95):.3f}s')
print(f'  有迟滞: {len(e_hyst)} 次, 时长 {[round(x,3) for x in e_hyst]} s, p95={np.percentile(e_hyst,95):.3f}s')
assert np.percentile(e_hyst, 95) > np.percentile(e_raw, 95), '迟滞让误报活得更久'
print('\n✅ **凡是存在此消彼长的一对机制，评测里必须同时有度量两端的指标** ——')
print('   只测闪烁不测误报时长，团队会一路把 τ_off 调到极低：闪烁一片绿，误报赖着三秒不走。')

## 6 · 回归门禁：显著性检验 + 多重比较校正 + 三档规则

20 个切片各做一次 α=0.05 的检验，即使模型没变，也有 64% 的概率至少出现一个假警报。
**Benjamini–Hochberg (BH-FDR) 是这个场景的正确工具**（Bonferroni 太保守）。

In [ ]:
def norm_cdf(z):
    return 0.5 * math.erfc(-z / math.sqrt(2.0))

def two_prop_pvalue(x_base, n_base, x_cand, n_cand):
    '''单边检验：candidate 的比例是否**显著低于** baseline。返回 p 值。'''
    if n_base == 0 or n_cand == 0:
        return 1.0
    p1, p2 = x_base / n_base, x_cand / n_cand
    p = (x_base + x_cand) / (n_base + n_cand)
    se = math.sqrt(max(p * (1 - p) * (1/n_base + 1/n_cand), 1e-18))
    return float(norm_cdf((p2 - p1) / se))

def bh_reject(pvals, alpha=0.05):
    '''Benjamini-Hochberg：返回被判为显著的下标集合。'''
    m = len(pvals)
    order = sorted(range(m), key=lambda i: pvals[i])
    k = 0
    for rank, i in enumerate(order, start=1):
        if pvals[i] <= alpha * rank / m:
            k = rank
    return sorted(order[:k])

# 教科书例子（Benjamini & Hochberg 1995 的经典 p 值列表）
PV = [0.001, 0.008, 0.039, 0.041, 0.042, 0.060, 0.074, 0.205, 0.212, 0.216]
assert bh_reject(PV, 0.05) == [0, 1], bh_reject(PV, 0.05)
assert len(bh_reject(PV, 0.20)) == 7, bh_reject(PV, 0.20)
bonf = [i for i, p in enumerate(PV) if p <= 0.05 / len(PV)]
assert bonf == [0], 'Bonferroni 只保留 1 个 —— 过于保守'
print(f'BH (α=0.05) 判为显著: {bh_reject(PV, 0.05)}   '
      f'BH (α=0.20): {bh_reject(PV, 0.20)}   Bonferroni (α=0.05): {bonf}')
print('✅ BH 控制的是「被判显著的结论中假阳性的比例」，比 Bonferroni 更适合几十个切片的场景。')

# 假警报率：20 个切片，模型完全没变
g = np.random.default_rng(17)
n_alarm_naive = n_alarm_bh = 0
for _ in range(300):
    ps = [two_prop_pvalue(int(g.binomial(200, 0.85)), 200,
                          int(g.binomial(200, 0.85)), 200) for _ in range(20)]
    n_alarm_naive += any(p <= 0.05 for p in ps)
    n_alarm_bh += len(bh_reject(ps, 0.05)) > 0
print(f'\n模型完全没变的 300 次重复实验，20 个切片：')
print(f'  不校正：{n_alarm_naive/300:.1%} 的批次至少出现一个「显著劣化」假警报')
print(f'  BH 校正：{n_alarm_bh/300:.1%}')
assert n_alarm_naive / 300 > 0.4, '不校正时假警报率应很高'
assert n_alarm_bh < n_alarm_naive
print('⚠️  一个天天误报的门禁，团队三周内就会学会绕过它 —— 这是门禁系统最常见的死法。')

In [ ]:
# ── 三档门禁判定器 ──
SLICES = [
    # name,                  级别,      baseline(hit, n),   candidate(hit, n), 容差
    ('critical/stop_yield',  'hard',    (209, 220), (196, 220), 0.00),
    ('critical/no_entry',    'hard',    (131, 140), (131, 140), 0.00),
    ('core/all_daylight',    'soft',    (2640, 3000), (2625, 3000), 0.01),
    ('core/all_night',       'soft',    (960, 1200), (948, 1200), 0.01),
    ('fm/small_far',         'soft',    (670, 1000), (620, 1000), 0.01),
    ('fm/occlusion',         'soft',    (192, 240), (190, 240), 0.01),
    ('fm/backlight',         'track',   (128, 180), (120, 180), 0.02),
    ('fm/rain_night',        'track',   (140, 200), (137, 200), 0.02),
]

def gate(slices, alpha=0.05):
    pvals = [two_prop_pvalue(b[0], b[1], c[0], c[1]) for _, _, b, c, _ in slices]
    sig = set(bh_reject(pvals, alpha))
    out, blocking = [], []
    for i, (name, level, b, c, tol) in enumerate(slices):
        rb, rc = b[0]/b[1], c[0]/c[1]
        drop = rb - rc
        if level == 'hard':
            fail = drop > tol + 1e-12
        elif level == 'soft':
            fail = (drop > tol + 1e-12) and (i in sig)
        else:
            fail = False
        out.append((name, level, rb, rc, drop, pvals[i], i in sig, fail))
        if fail:
            blocking.append(name)
    return out, blocking

rep, blocking = gate(SLICES)
print(f"{'切片':<22s} {'档':<6s} {'base':>7s} {'cand':>7s} {'跌幅':>7s} "
      f"{'p值':>8s} {'BH显著':>7s} {'判定'}")
for name, lv, rb, rc, dr, pv, s, f in rep:
    print(f'{name:<22s} {lv:<6s} {rb:>7.3f} {rc:>7.3f} {dr:>+7.3f} '
          f'{pv:>8.3f} {"是" if s else "否":>7s} {"❌ 阻断" if f else "✅ 通过"}')
print(f'\n阻断项: {blocking}')
assert 'critical/stop_yield' in blocking, '关键类掉了 13 个样本，硬门禁必须拦住'
assert 'critical/no_entry' not in blocking, '没跌就不该拦（硬门禁不是「一律拦」）'
assert 'fm/small_far' in blocking, '软门禁：跌幅超容差 + BH 显著 -> 阻断'
assert 'core/all_night' not in blocking, '软门禁：跌幅未超容差 -> 通过'
assert 'fm/backlight' not in blocking, '观察项不阻断（哪怕掉了 4.4 个点）'
assert all(not r[7] for r in rep if r[1] == 'track')
print('\n✅ 三档设计的意义：硬门禁少而绝对（关键类、FP 上限、p99 延迟）；')
print('   软门禁需要**同时满足「跌幅超容差」和「统计显著」**；观察项只记趋势。')
print('⚠️  容差必须来自**种子方差**（跑 3-5 个种子测自然波动，取 2σ），不能拍脑袋。')
print('⚠️  门禁必须可 override，但每次覆盖要记录理由与责任人 ——')
print('   同一条门禁被覆盖三次，要么阈值定错了，要么你们有一个一直没修的真问题。')

## 7 · 离线-路测背离：分母不匹配到底让你低估了多少

评测集里 80% 的帧含标志，真实道路只有 5%。
而误检主要发生在**无标志的背景帧**上 —— 这类帧在评测集里被严重欠采样。

In [ ]:
Q_BG, Q_SIGN = 0.004, 0.001          # 背景帧 / 含标志帧的每帧误检率
def fp_per_frame(pi_bg):
    return pi_bg * Q_BG + (1 - pi_bg) * Q_SIGN

PI_EVAL, PI_ROAD = 0.20, 0.95        # 评测集 vs 真实道路的背景帧占比
KM, HOURS = 60.0, 1.0
FRAMES_ROAD = HOURS * 3600 * 30

fp_eval_rate = fp_per_frame(PI_EVAL)
fp_road_rate = fp_per_frame(PI_ROAD)
naive_fp = fp_eval_rate * FRAMES_ROAD          # 天真外推
true_fp = fp_road_rate * FRAMES_ROAD

print(f"{'':<26s} {'背景帧占比':>10s} {'FP/帧':>10s} {'外推到 60km 的 FP 数':>20s} {'FP/km':>8s}")
print(f'{"评测集（天真外推）":<26s} {PI_EVAL:>10.2f} {fp_eval_rate:>10.5f} '
      f'{naive_fp:>20.0f} {naive_fp/KM:>8.2f}')
print(f'{"真实道路":<26s} {PI_ROAD:>10.2f} {fp_road_rate:>10.5f} '
      f'{true_fp:>20.0f} {true_fp/KM:>8.2f}')
ratio = true_fp / naive_fp
print(f'\n低估倍数 = {ratio:.2f}×   ← 「离线看起来达标」的版本会在路测第一天被否掉')
assert ratio > 2.0, f'低估倍数应超过 2 倍，得到 {ratio:.2f}'
assert abs(fp_eval_rate - 0.0016) < 1e-9 and abs(fp_road_rate - 0.00385) < 1e-9

# 修法②：按真实 π_bg 对两类帧重新加权
def reweighted_fp_rate(n_bg_eval, fp_bg, n_sign_eval, fp_sign, pi_bg_road):
    q_bg = fp_bg / max(n_bg_eval, 1); q_sign = fp_sign / max(n_sign_eval, 1)
    return pi_bg_road * q_bg + (1 - pi_bg_road) * q_sign

n_bg_e, n_sg_e = 400, 1600
fp_bg_e, fp_sg_e = n_bg_e * Q_BG, n_sg_e * Q_SIGN
rw = reweighted_fp_rate(n_bg_e, fp_bg_e, n_sg_e, fp_sg_e, PI_ROAD)
print(f'\n重加权后的估计: {rw:.5f} FP/帧  (真值 {fp_road_rate:.5f})  '
      f'误差 {abs(rw-fp_road_rate)/fp_road_rate:.1%}')
assert abs(rw - fp_road_rate) < 1e-9, '重加权应精确还原真实 FP 率'

# 帧级 vs 事件级：用户感受的是「几次」，不是「几帧」
AVG_EPISODE_FRAMES = 9
print(f'\n帧级: {true_fp:.0f} 个 FP **帧** / 小时')
print(f'事件级: {true_fp/AVG_EPISODE_FRAMES:.0f} 次 FP **事件** / 小时'
      f'（每次约 {AVG_EPISODE_FRAMES/30:.2f} s）')
assert true_fp / AVG_EPISODE_FRAMES < true_fp / 5
print('\n✅ 修法①（唯一诚实的做法）：显式保留一批**连续里程片段**，')
print('   哪怕大部分帧没有标志，专门用来测 FP/km。')
print('✅ 修法②：按真实 π_bg 对两类帧重新加权再汇总。')
print('⚠️  这个偏差是**结构性的**：标注是按「有目标的片段」组织的，')
print('   标注供应商没有动力去标一万帧空旷的高速路 —— 评测集天然过采样含标志帧。')

## ✏️ 练习 1：COCO 风格的 mAP@[.5:.95]

实现 `coco_map(gt_boxes, dets, thrs=None)`：在 IoU 阈值 `0.50, 0.55, ..., 0.95`（共 10 档）
上分别算 AP 再取平均。可以复用 `match_frame` 与 `average_precision`。

In [ ]:
def coco_map(gt_boxes, dets, thrs=None):
    # TODO: thrs 默认 [0.50, 0.55, ..., 0.95]（10 档）
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
g1 = [np.array([0., 0., 100., 100.])]
perfect = [(np.array([0., 0., 100., 100.]), 0.9)]
assert abs(coco_map(g1, perfect) - 1.0) < 1e-9, '完美定位在所有阈值上都应是 1.0'

# 构造 IoU 恰好 = 0.6 的预测：交 = 0.6 * 并
#   宽 100 的框平移 d：IoU = (100-d)/(100+d) = 0.6  ->  d = 25
d = 25.0
iou06 = [(np.array([d, 0., 100. + d, 100.]), 0.9)]
assert abs(iou(g1[0], iou06[0][0]) - 0.6) < 1e-9, iou(g1[0], iou06[0][0])
m = coco_map(g1, iou06)
# 只在 0.50 / 0.55 / 0.60 三档上算命中 -> 3/10 = 0.3
assert abs(m - 0.3) < 1e-9, f'IoU=0.6 的预测 mAP@[.5:.95] 应为 0.3，得到 {m}'
assert abs(coco_map(g1, iou06, thrs=[0.5]) - 1.0) < 1e-9, 'AP@0.5 应为 1.0'
assert abs(coco_map(g1, iou06, thrs=[0.65]) - 0.0) < 1e-9, 'AP@0.65 应为 0.0'
assert coco_map(g1, []) == 0.0, '没有任何预测时应为 0.0'

print(f'完美定位 mAP@[.5:.95] = {coco_map(g1, perfect):.3f}')
print(f'IoU=0.60 mAP@[.5:.95] = {m:.3f}   （AP@0.5 却是 1.000）')
print('✅ 练习 1 通过：**COCO mAP 把一半权重压在 IoU>0.75 上** ——')
print('   对 TSR 而言那段区间没有下游价值，用它做主指标是把优化力气引向错误的方向。')

## ✏️ 练习 2：代价加权风险分

实现 `risk(confusion, cost_miss, cost_fp, cost_conf, dist_km)`，其中
`confusion` 是 `{(c_true, c_pred): n}`，`c_pred` 为 `None` 表示漏检、
`c_true` 为 `None` 表示误检。返回 **每 100 km 的风险分**。
未在 `cost_conf` 中列出的错分对，代价取 `max(cost_miss[c_true], cost_fp[c_pred]) * 0.6`。

In [ ]:
def risk(confusion, cost_miss, cost_fp, cost_conf, dist_km):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
CM = {'stop': 100.0, 'sp60': 8.0, 'sp80': 8.0}
CF = {'stop': 20.0, 'sp60': 12.0, 'sp80': 12.0}
CC = {('sp60', 'sp80'): 15.0, ('sp80', 'sp60'): 2.0}

assert abs(risk({('stop', None): 1}, CM, CF, CC, 100.0) - 100.0) < 1e-9
assert abs(risk({(None, 'sp60'): 2}, CM, CF, CC, 100.0) - 24.0) < 1e-9
assert abs(risk({('sp60', 'sp80'): 1}, CM, CF, CC, 100.0) - 15.0) < 1e-9
assert abs(risk({('sp80', 'sp60'): 1}, CM, CF, CC, 100.0) - 2.0) < 1e-9, '方向不对称'
# 未列出的错分对：max(100, 12) * 0.6 = 60
assert abs(risk({('stop', 'sp60'): 1}, CM, CF, CC, 100.0) - 60.0) < 1e-9
# 里程归一化：同样的错误跑 50 km，风险分翻倍
assert abs(risk({('stop', None): 1}, CM, CF, CC, 50.0) - 200.0) < 1e-9
# 正确预测不计代价
assert abs(risk({('sp60', 'sp60'): 999}, CM, CF, CC, 100.0)) < 1e-9

mix = {('stop', None): 2, (None, 'sp80'): 3, ('sp60', 'sp80'): 4, ('sp80', 'sp60'): 4}
print(f'混合场景风险分 = {risk(mix, CM, CF, CC, 100.0):.1f} / 100km')
assert abs(risk(mix, CM, CF, CC, 100.0) - (200 + 36 + 60 + 8)) < 1e-9
print('✅ 练习 2 通过：**同一对类别，两个方向的代价可以差 7.5 倍** ——')
print('   这一层不对称是 mAP 结构上无法表达的。')

## ✏️ 练习 3：时序稳定性指标

实现 `temporal_report(report_flags, ranges, fps=30.0, stable_ratio=0.95)`，返回
`{'first_range', 'stable_range', 'flips', 'on_ratio', 'longest_off_s'}`。
`longest_off_s` 是**已上报之后**出现的最长连续未上报时长（秒）——它度量「输出闪断」。
若从未上报，`first_range` / `stable_range` 为 `None`，其余为 0。

In [ ]:
def temporal_report(report_flags, ranges, fps=30.0, stable_ratio=0.95):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
rg = [100.0, 90.0, 80.0, 70.0, 60.0, 50.0, 40.0, 30.0]
f1 = [False, True, False, False, True, True, True, True]
r1 = temporal_report(f1, rg)
assert r1['first_range'] == 90.0, r1
assert r1['stable_range'] == 60.0, f"从 60m 起 4/4 都上报，得到 {r1['stable_range']}"
assert r1['flips'] == 3, r1                       # F->T, T->F, F->T
assert abs(r1['on_ratio'] - 5/8) < 1e-9
assert abs(r1['longest_off_s'] - 2/30) < 1e-9, '首次上报后最长断了 2 帧'

f2 = [False] * 8
r2 = temporal_report(f2, rg)
assert r2['first_range'] is None and r2['stable_range'] is None
assert r2['flips'] == 0 and r2['on_ratio'] == 0.0 and r2['longest_off_s'] == 0.0

f3 = [True] * 8
r3 = temporal_report(f3, rg)
assert r3['first_range'] == 100.0 and r3['stable_range'] == 100.0
assert r3['flips'] == 0 and r3['longest_off_s'] == 0.0

for nm, f in [('闪断型', f1), ('全无', f2), ('完美', f3)]:
    r = temporal_report(f, rg)
    print(f'{nm:<6s} first={r["first_range"]} stable={r["stable_range"]} '
          f'flips={r["flips"]} on={r["on_ratio"]:.2f} longest_off={r["longest_off_s"]:.3f}s')
print('✅ 练习 3 通过：**首报距离、稳定距离、闪断时长三个数必须一起看** ——')
print('   任何一个单独拿出来都能被优化成好看的样子。')

## ✏️ 练习 4：多重比较校正下的门禁判定

实现 `gate_decide(slices, alpha=0.05)`：`slices` 每项为
`(name, level, (hit_base, n_base), (hit_cand, n_cand), tol)`，`level ∈ {'hard','soft','track'}`。
用 `two_prop_pvalue` + `bh_reject` 做校正，返回 `(报告列表, 阻断项名单)`：
- `hard`：跌幅 > tol 就阻断（**不看显著性**——关键类不能等统计显著）
- `soft`：跌幅 > tol **且** BH 判为显著才阻断
- `track`：永不阻断

In [ ]:
def gate_decide(slices, alpha=0.05):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
S = [('crit/stop',  'hard',  (100, 100), (99, 100), 0.00),   # 跌 0.01 > 0 -> 阻断
     ('crit/entry', 'hard',  (100, 100), (100, 100), 0.00),  # 没跌 -> 通过
     ('core/day',   'soft',  (900, 1000), (880, 1000), 0.005),
     ('core/night', 'soft',  (900, 1000), (897, 1000), 0.005),  # 跌 0.003 < tol -> 通过
     ('fm/backlit', 'track', (50, 100), (30, 100), 0.00)]    # 掉惨了但只是观察项
rep_, block_ = gate_decide(S)
assert 'crit/stop' in block_ and 'crit/entry' not in block_
assert 'fm/backlit' not in block_, '观察项永不阻断'
assert 'core/night' not in block_, '跌幅在容差内不阻断'
assert len(rep_) == len(S)

# hard 档不看显著性：只跌 1 个样本，p 值远不显著，但仍必须阻断
p_stop = two_prop_pvalue(100, 100, 99, 100)
assert p_stop > 0.05, f'该切片统计上不显著（p={p_stop:.3f}），但硬门禁仍要拦'
print(f'crit/stop 的 p 值 = {p_stop:.3f}（不显著），但硬门禁仍然阻断 ✅')

# soft 档：同样的跌幅，样本量大才会被判显著
big = [('a', 'soft', (8500, 10000), (8400, 10000), 0.005)]
small = [('a', 'soft', (85, 100), (84, 100), 0.005)]
assert gate_decide(big)[1] == ['a'], '大样本 -> 显著 -> 阻断'
assert gate_decide(small)[1] == [], '小样本 -> 不显著 -> 不阻断'
print('大样本切片跌 0.01 -> 阻断；小样本切片跌 0.01 -> 不阻断（证据不足）✅')
print(f'\n本次阻断项: {block_}')
print('✅ 练习 4 通过：**关键类的硬门禁不能等统计显著** ——')
print('   等到 40 个 stop 样本里的差异「统计显著」，事故已经发生很多次了。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def coco_map(gt_boxes, dets, thrs=None):
    if thrs is None:
        thrs = [0.50 + 0.05 * i for i in range(10)]
    aps = []
    for t in thrs:
        tp = match_frame(gt_boxes, dets, iou_thr=t)
        aps.append(average_precision([d[1] for d in dets], tp, len(gt_boxes)))
    return float(np.mean(aps))

In [ ]:
# 练习 2 参考答案
def risk(confusion, cost_miss, cost_fp, cost_conf, dist_km):
    total = 0.0
    for (ct, cp), n in confusion.items():
        if ct == cp:
            continue                                   # 正确预测
        if cp is None:                                 # 漏检
            total += n * cost_miss[ct]
        elif ct is None:                               # 误检
            total += n * cost_fp[cp]
        elif (ct, cp) in cost_conf:                    # 显式列出的错分对
            total += n * cost_conf[(ct, cp)]
        else:                                          # 未列出：取较重一侧打折
            total += n * max(cost_miss[ct], cost_fp[cp]) * 0.6
    return total / dist_km * 100.0

In [ ]:
# 练习 3 参考答案
def temporal_report(report_flags, ranges, fps=30.0, stable_ratio=0.95):
    n = len(report_flags)
    on_idx = [i for i, f in enumerate(report_flags) if f]
    if not on_idx:
        return {'first_range': None, 'stable_range': None,
                'flips': 0, 'on_ratio': 0.0, 'longest_off_s': 0.0}
    first_range = max(ranges[i] for i in on_idx)
    stable_range, suffix = None, 0
    for i in range(n - 1, -1, -1):                     # 从后往前找最远的稳定起点
        suffix += report_flags[i]
        if suffix / (n - i) >= stable_ratio:
            stable_range = ranges[i]
    flips = sum(1 for i in range(1, n) if report_flags[i] != report_flags[i-1])
    longest, run = 0, 0
    for i in range(on_idx[0], n):                      # 只统计首次上报之后
        run = 0 if report_flags[i] else run + 1
        longest = max(longest, run)
    return {'first_range': first_range, 'stable_range': stable_range,
            'flips': flips, 'on_ratio': len(on_idx) / n,
            'longest_off_s': longest / fps}

In [ ]:
# 练习 4 参考答案
def gate_decide(slices, alpha=0.05):
    pvals = [two_prop_pvalue(b[0], b[1], c[0], c[1]) for _, _, b, c, _ in slices]
    sig = set(bh_reject(pvals, alpha))
    report_, blocking = [], []
    for i, (name, level, b, c, tol) in enumerate(slices):
        rb, rc = b[0] / b[1], c[0] / c[1]
        drop = rb - rc
        if level == 'hard':
            fail = drop > tol + 1e-12                  # 关键类不等统计显著
        elif level == 'soft':
            fail = (drop > tol + 1e-12) and (i in sig)
        else:
            fail = False
        report_.append({'name': name, 'level': level, 'base': rb, 'cand': rc,
                        'drop': drop, 'p': pvals[i], 'significant': i in sig,
                        'blocked': fail})
        if fail:
            blocking.append(name)
    return report_, blocking

---
## 🧪 真实工程胶囊：一份可直接落地的 TSR 评测记分卡与流水线

In [ ]:
RECIPE = r'''
# ═══════════════════════════════════════════════════════════════════
# TSR 评测记分卡 scorecard.json —— 结构模板
# 不变量：**「存在一份 scorecard.json」本身就编码了「该版本通过了评测」**
#         没有 scorecard 的产物一律不允许进入发布流程。
# ═══════════════════════════════════════════════════════════════════
{
  "provenance": {                       # ← 可复现性的全部输入，缺一不可
    "model_commit": "a3f9c1e", "config_hash": "sha256:8c21...",
    "dataset_version": "tsr_eval_v7.2", "eval_code_commit": "b71d0aa",
    "seed": 0, "tta": false, "postproc": {"score_thr": 0.35, "nms_iou": 0.6},
    "runtime": "trt-8.6 / orin-x / fp16", "timestamp": "2026-08-17T09:12:03Z"
  },

  "primary": {                          # ← **唯一的优化目标**，用于排序模型
    "risk_score_per_100km": 21.4, "baseline": 23.9, "delta": -2.5
  },

  "guardrails": [                       # ← 硬约束，任一超限直接阻断发布
    {"name": "fp_per_100km_overall",      "value": 0.62, "limit": 1.0,  "ok": true},
    {"name": "fp_per_100km_critical",     "value": 0.004,"limit": 0.01, "ok": true},
    {"name": "per_sign_recall_stop_yield","value": 0.968,"min": 0.965,  "ok": true},
    {"name": "first_report_range_p50_m",  "value": 72.4, "min": 70.0,   "ok": true},
    {"name": "flicker_per_sign",          "value": 0.31, "limit": 1.0,  "ok": true},
    {"name": "fp_episode_p95_s",          "value": 0.43, "limit": 0.5,  "ok": true},
    {"name": "latency_p99_ms",            "value": 11.8, "limit": 15.0, "ok": true}
  ],

  "slices": [                           # ← 诊断层：不进门禁，用于**解释**主指标
    {"name": "size/<16px",  "n_pos": 412, "ap": 0.512, "ci95": [0.471, 0.556]},
    {"name": "size/16-32px","n_pos": 1180,"ap": 0.804, "ci95": [0.788, 0.821]},
    {"name": "dist/>80m",   "n_pos": 290, "ap": 0.441, "ci95": [0.392, 0.489]},
    {"name": "light/night", "n_pos": 860, "ap": 0.712, "ci95": [0.690, 0.735]},
    {"name": "night x >80m","n_pos": 96,  "ap": 0.310, "ci95": [0.230, 0.395]}
  ],

  "gate": {"alpha": 0.05, "correction": "benjamini-hochberg",
           "blocking": [], "overrides": []},   # override 必须记录理由 + 责任人
  "verdict": "PASS"
}

# ═══════════════════════════════════════════════════════════════════
# 评测流水线（每一步的产物与门禁条件）
# ═══════════════════════════════════════════════════════════════════
# ① 推理     固定 seed / 关 TTA / 固定后处理参数 / 记录 runtime
#            产物: raw_predictions.jsonl（**必须归档** —— 指标口径变了要能重算）
# ② 指标     评测代码有单元测试（完美预测=1.0 / 全空=0.0 / 手算例子）
#            产物: metrics_raw.json
# ③ 切片     分桶 x 类别 x 时序，**全部带 bootstrap CI**
#            规则: 每桶 >= 50 正样本，否则降级为「观察项」
# ④ 对比     配对检验 + BH-FDR 多重比较校正（20 个切片不校正 = 64% 假警报率）
#            容差来自**种子方差**：跑 3-5 个种子测自然波动，取 2σ
# ⑤ 门禁     hard（不等显著性）/ soft（跌幅+显著性）/ track（只记趋势）
#            产物: scorecard.json  → 通过才允许打包发布
#
# ═══════════════════════════════════════════════════════════════════
# 连续里程片段（唯一诚实测 FP/km 的方式）
# ═══════════════════════════════════════════════════════════════════
# continuous_drives/          **大部分帧没有标志，这正是重点**
#   ├── highway_day_120km/    背景帧占比 ~0.96，接近真实分布
#   ├── urban_night_40km/     广告牌密集，FP 主战场
#   └── tunnel_series_15km/   出入口过曝/欠曝
# 用途：只算 FP/km、FP/hour、误报事件数与持续时长；**不算 mAP**（正样本太少）
#
# ⚠️ 常规评测集会过采样含标志帧（π_bg≈0.2 vs 真实 0.95）
#    -> 天真外推会把 FP/km 低估约 2.4 倍
#    -> 修法①用连续里程片段  修法②按真实 π_bg 重加权
'''
print(RECIPE)
for key in ['provenance', 'risk_score_per_100km', 'fp_per_100km_critical',
            'first_report_range_p50_m', 'fp_episode_p95_s', 'benjamini-hochberg',
            'ci95', 'raw_predictions.jsonl', 'continuous_drives', '2σ']:
    assert key in RECIPE, key
print('✅ 配方覆盖：可复现性字段 / 主指标 / 7 条护栏 / 分桶+CI / BH 校正 / '
      '三档门禁 / 连续里程片段 / 分母不匹配')

### 小结

- **mAP 有五个盲区**：①按类别平均把关键类稀释（一个类掉 20 点，60 类 mAP 只掉 0.33，
  落在种子噪声内）②与距离/尺寸无关 ③完全没有时序（闪 15 次/秒和稳定输出的 mAP 一样）
  ④precision 是相对量，分母是系统自己的输出、**可以靠提阈值刷** ⑤与下游动作脱节
  （框准 0.05 IoU 没价值，60 认成 80 是灾难）。
  **mAP 是迭代信号，不是发布判据。**
- **分桶是解药，但有统计代价**：每桶 ≥50 正样本、所有分桶指标必须带 bootstrap CI；
  显式建立 3–5 个安全关键的**交叉桶**（夜间×远距）；
  **micro/macro 的选择会翻转模型排名**，TSR 默认用 macro。
- **代价三层不对称**：类别之间、错误方向之间（60→80 比 80→60 危险 7.5 倍）、
  漏检与误检之间。写成代价矩阵后，**风险分（代价/100km）替代 mAP 成为唯一主指标**，
  且能逐项归因。代价数值要从 HARA/FMEA 反推，有文档有版本号。
- **换分母**：precision → **FP per km**（分母是里程，不可刷、可跨系统比、直接对应体验，
  量产目标 FP/100km < 1）；per-frame recall → **per-sign recall + 首检距离分布**。
  判据：分母是系统自己的输出就会被污染，分母是外生物理量才诚实。
- **时序稳定性五指标**：首次**上报**距离 / 稳定检出距离 / 闪烁次数 / 误报持续时长 p95 /
  类别跳变率。**凡是存在此消彼长的一对机制，评测里必须同时度量两端**，否则优化会滑向一边。
  前提：标注必须有物理标志级的 instance ID —— **评测指标决定标注规范**。
- **门禁三档**：hard（关键类、FP 上限、p99 延迟——**不等统计显著**）/
  soft（跌幅 + BH 显著）/ track（只记趋势）。容差来自种子方差（2σ）；
  20 个切片不做多重比较校正会有 64% 的假警报率；**门禁必须可 override 但要留记录**。
- **离线-路测背离七因**，头号是**分母不匹配**：评测集 π_bg≈0.2 而真实道路 ≈0.95，
  天真外推把 FP/km **低估约 2.4 倍**。唯一诚实的修法是保留一批**连续里程片段**。
- **「Build and maintain evaluation pipelines」的满分答案骨架**：
  ①评测产出的是决策不是数字（主指标/护栏/诊断的分层）
  ②评测本身是要被测试的软件（单测、数据版本、raw prediction 归档）
  ③评测要贴着线上分布（分母不匹配、连续里程片段、偏差的量化）。

本课到此结束。回看整门课：模块 01 定义了数据与类别体系，02 定了系统架构，
03 列全了失效模式，04 把单帧变成稳定输出，05 给出了判断这一切好坏的尺子。
**下一步去 C57（小目标）与 C58（数据闭环）—— 它们回答的是「指标不够好时该做什么」。**